# Vectorization Techniques for NLP: From Bag-of-Words to Word Embeddings

Machine learning models only understand numbers, but text is made of words. **Vectorization** is the process of turning text into numeric vectors a model can actually use. This notebook walks through that story in order of increasing sophistication:

| # | Technique | Type | Captures word order? | Captures meaning/similarity? |
|---|---|---|---|---|
| 1 | Count Vectorization (Bag-of-Words) | Sparse, count-based | No | No |
| 2 | N-gram Vectorization | Sparse, count-based | Partially (local order) | No |
| 3 | TF-IDF | Sparse, weighted | No | No (but weights by importance) |
| 4 | Word2Vec | Dense, learned | No (uses context to train) | Yes |
| 5 | FastText | Dense, learned (subword-aware) | No | Yes, plus handles unseen words |
| 6 | Pretrained embeddings (GloVe) | Dense, pretrained | No | Yes, from a huge corpus |

We'll use **one running example corpus throughout** (a handful of short movie-review-style sentences) so you can see exactly how the *same* text gets turned into very different kinds of vectors by each technique, and compare the outputs directly.




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Our running example corpus

A small set of movie-review-style sentences — short, relatable, and with a clear mix of positive words (*fantastic, wonderful, great, love*) and negative words (*terrible, boring, bad, ruined*). 


In [ ]:
documents = [
    "I love this movie, the story was fantastic and heartwarming",
    "This movie was terrible, the story made no sense at all",
    "The acting was great but the story was a bit boring",
    "What a fantastic movie! Great acting and a wonderful story",
    "Terrible acting ruined an otherwise decent story",
    "I really love the story and the acting was wonderful",
    "The movie was boring, slow, and the story dragged on forever",
    "Great story, great acting, this was a truly fantastic movie",
]

doc_labels = [f"Doc {i+1}" for i in range(len(documents))]

for label, doc in zip(doc_labels, documents):
    print(f"{label}: {doc}")


### **1. Count Vectorization (Bag-of-Words)**

The simplest possible approach: build a vocabulary of every unique word across all documents, then represent each document as a vector of **how many times each word appears**. This is called "Bag-of-Words" because it treats a document as an unordered bag of words — *"great acting, terrible story"* and *"terrible acting, great story"* produce the **exact same vector**, even though they mean opposite things. Word order is completely discarded.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X = cv.fit_transform(documents)

print("Shape (documents x vocabulary size):", X.shape)


In [ ]:
bow_df = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=doc_labels)
bow_df


In [ ]:
# A few words to spot-check by eye
bow_df[['movie', 'story', 'fantastic', 'terrible', 'the']]


Notice: `the` shows up with a fairly high count in several documents just because it's a common English word, not because it carries any special meaning about the review. That's the core weakness of plain Bag-of-Words that **TF-IDF** (below) is designed to fix.

**Limitations of Bag-of-Words:**
- Word order is thrown away entirely.
- Common words (*the, was, a*) get the same treatment as meaningful words (*fantastic, terrible*).
- Vocabulary size — and therefore vector length — grows with every new unique word in the corpus, which becomes a problem at real-world scale (hundreds of thousands of words).


### **2. N-gram Vectorization**

Bag-of-Words treats every word independently, so it can't tell *"good"* from *"not good"* — both documents would simply contain the words `not`, `good` with no link between them. **N-grams** partially fix this by treating groups of `N` consecutive words as a single vocabulary item:

- **Unigrams** (`n=1`): individual words — plain Bag-of-Words.
- **Bigrams** (`n=2`): pairs of consecutive words, e.g. `"not good"`.
- **Trigrams** (`n=3`): triples of consecutive words, e.g. `"was not good"`.

Let's make this concrete with a tiny, deliberately minimal example before returning to the full movie-review corpus.


In [ ]:
negation_example = [
    "the movie was good",
    "the movie was not good",
]

# Unigrams only (ngram_range=(1,1)) - the default - cannot distinguish these well:
uni = CountVectorizer(ngram_range=(1, 1))
X_uni = uni.fit_transform(negation_example)
pd.DataFrame(X_uni.toarray(), columns=uni.get_feature_names_out(), index=['Positive', 'Negated'])


In [ ]:
# Unigrams + bigrams (ngram_range=(1,2)) now captures 'was good' vs 'was not' / 'not good':
bi = CountVectorizer(ngram_range=(1, 2))
X_bi = bi.fit_transform(negation_example)
pd.DataFrame(X_bi.toarray(), columns=bi.get_feature_names_out(), index=['Positive', 'Negated'])


With unigrams alone, both sentences share every single word (`good` appears in both, so does `movie`, `the`, `was`) — a unigram-only model has almost no way to tell them apart. Once we add bigrams, the columns `was good` vs. `was not` and `not good` appear only in the negated sentence — exactly the signal we were missing.

Now let's apply n-grams to our full running corpus, up to trigrams, matching the original exercise:


In [ ]:
ngram = CountVectorizer(ngram_range=(1, 3))
X_ngram = ngram.fit_transform(documents)

print("Shape:", X_ngram.shape)
ngram_df = pd.DataFrame(X_ngram.toarray(), columns=ngram.get_feature_names_out(), index=doc_labels)
ngram_df.iloc[:, :10]


**The cost of n-grams:** vocabulary size grows fast as `N` increases, since the number of *possible* word combinations explodes combinatorially. Let's actually measure that growth on our corpus:

In [ ]:
vocab_sizes = {}
for n in [1, 2, 3, 4]:
    vec = CountVectorizer(ngram_range=(1, n))
    vec.fit(documents)
    vocab_sizes[f'(1,{n})'] = len(vec.get_feature_names_out())

vocab_sizes_series = pd.Series(vocab_sizes)
print(vocab_sizes_series)

fig, ax = plt.subplots(figsize=(7, 5))
vocab_sizes_series.plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_xlabel('ngram_range')
ax.set_ylabel('Vocabulary size')
ax.set_title('Vocabulary size grows quickly with N (on just 8 short sentences!)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


On a real corpus with thousands of documents, going from unigrams to trigrams can take a vocabulary from tens of thousands of columns to millions — this is the practical reason n-grams are usually capped at 2 or 3, and often combined with a minimum document-frequency filter (`min_df=...`) to drop extremely rare combinations.


### **3. TF-IDF Vectorization**

TF-IDF (**T**erm **F**requency - **I**nverse **D**ocument **F**requency) fixes Bag-of-Words' biggest weakness: it downweights words that appear in *many* documents (like `the`, `was`, `movie` — not very informative about what makes any *one* review distinctive) and upweights words that are frequent in one document but rare across the whole corpus (much more informative).

$$\text{tfidf}(w, d) = \underbrace{\text{count}(w, d)}_{\text{term frequency}} \times \underbrace{\ln\!\left(\frac{1 + N}{1 + \text{df}(w)}\right) + 1}_{\text{inverse document frequency}}$$

where $N$ is the number of documents and $\text{df}(w)$ is the number of documents containing word $w$ (scikit-learn's exact smoothed formula; the intuition is the same across implementations: **rare-across-corpus words get a higher weight**).


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(documents)

print("Shape:", X_tfidf.shape)
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out(), index=doc_labels).round(3)
tfidf_df[['the', 'movie', 'fantastic', 'terrible']]


Compare this to the raw counts for the same four words earlier: **`the`** and **`movie`** get relatively small TF-IDF weights everywhere (they show up all over the corpus, so they're not distinctive), while **`fantastic`** and **`terrible`** — which each appear in only a couple of documents — get much larger weights exactly in the documents where they occur. That's TF-IDF doing its job: separating "frequent everywhere" from "frequent and distinctive."

**A practical use of TF-IDF: automatic keyword extraction.** The words with the highest TF-IDF score in a document are a reasonable first guess at that document's "keywords" — a technique genuinely used for search indexing and quick document summarization. Let's try it, this time also removing common English stopwords (`the`, `was`, `a`, ...) so the keywords come out cleaner.


In [ ]:
tfidf_clean = TfidfVectorizer(stop_words='english')
X_clean = tfidf_clean.fit_transform(documents)
tfidf_clean_df = pd.DataFrame(X_clean.toarray(), columns=tfidf_clean.get_feature_names_out(), index=doc_labels)

for label, row in tfidf_clean_df.iterrows():
    top_terms = row.sort_values(ascending=False).head(3)
    top_terms = top_terms[top_terms > 0]
    print(f"{label}: {list(top_terms.index)}")


Notice how much more meaningful these keywords are compared to what you'd get picking the top-3 *raw counts* per document (which would mostly just return `the`, `was`, `a` in every single document) — this is the concrete payoff of TF-IDF weighting over plain Bag-of-Words.


### **4. Word2Vec Vectorization**

Everything so far has been **sparse and count-based**: a word either appears or it doesn't, and the vectors have no notion that `fantastic` and `wonderful` mean similar things — to Bag-of-Words/TF-IDF, `fantastic` and `bicycle` are just as unrelated as `fantastic` and `wonderful`.

**Word2Vec** (Mikolov et al., 2013) fixes this by learning a **dense, low-dimensional vector for every word** from its surrounding context, based on the *distributional hypothesis*: *"a word is characterized by the company it keeps."* Two training strategies exist:

- **CBOW (Continuous Bag-of-Words)** — predict a word from its surrounding context words. Faster, works well with frequent words.
- **Skip-gram** — predict the surrounding context words from a single word. Slower, but tends to work better for rare words.

`gensim`'s `sg` parameter controls this: `sg=0` → CBOW, `sg=1` → Skip-gram.

> **Note on scale:** Word2Vec needs a *lot* of text to learn genuinely good vectors — production Word2Vec models are trained on billions of words. Our 8-sentence corpus is far too small to learn reliable semantics; we're using it here purely to show the *mechanics* of training and querying a Word2Vec model. Later in this notebook we'll load a real pretrained model (GloVe) trained on billions of words, where the semantic relationships actually hold up.


In [ ]:
import re

def simple_tokenize(text):
    """Lowercase and split on non-letter characters - a small, dependency-free tokenizer."""
    return re.findall(r"[a-z']+", text.lower())

# Fix: the original notebook overwrote `sample_text` in place while looping over it (mutating
# the list you're iterating over, and destroying the original sentences you'd need again later).
# We build a NEW list instead, leaving `documents` untouched and reusable.
tokenized_docs = [simple_tokenize(doc) for doc in documents]

for tokens in tokenized_docs:
    print(tokens)


In [ ]:
from gensim.models import Word2Vec


w2v_model = Word2Vec(
    tokenized_docs,
    vector_size=8,   # dimensionality of each word vector (kept small - our corpus is tiny)
    window=3,        # how many neighboring words count as "context"
    min_count=1,      # keep every word, even ones seen only once (fine for a toy corpus this small)
    sg=0,             # 0 = CBOW, 1 = Skip-gram
    workers=1,
)

print("Vocabulary learned:", list(w2v_model.wv.index_to_key))


In [ ]:
# Fix: `.most_similar('sport')[0]` only ever prints ONE result and silently assumes the word
# exists in the vocabulary. Here we print the full ranked list, for a word that's actually in
# OUR corpus's vocabulary ('story').
similar_words = w2v_model.wv.most_similar('story')
pd.DataFrame(similar_words, columns=['word', 'cosine similarity'])


On such a tiny training corpus, don't expect these similarity rankings to be deeply meaningful yet — that's expected, and exactly the limitation flagged above. What *is* worth confirming is that the mechanics work: every word in the vocabulary now has an 8-number vector, learned purely from which words tend to appear near which other words in our 8 sentences.


In [ ]:
# Visualize the learned word vectors in 2D (PCA reduces our 8-dim vectors down to 2 dims
# purely for plotting - it does not change the underlying model).
from sklearn.decomposition import PCA

words = list(w2v_model.wv.index_to_key)
vectors = w2v_model.wv[words]

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(coords[:, 0], coords[:, 1], color='#4C72B0')
for word, (x, y) in zip(words, coords):
    ax.annotate(word, (x, y), fontsize=11, xytext=(4, 4), textcoords='offset points')
ax.set_title('Word2Vec embeddings, reduced to 2D with PCA')
ax.set_xlabel('PCA component 1')
ax.set_ylabel('PCA component 2')
plt.tight_layout()
plt.show()


### Using a trained Word2Vec model as a Keras `Embedding` layer

The original notebook used `model.wv.get_keras_embedding(...)`, which has become unreliable across recent `gensim` + `Keras` version combinations (the method was effectively dropped as `gensim` and `Keras` evolved independently). The currently-recommended approach ([gensim's own documentation](https://github.com/RaRe-Technologies/gensim/wiki/Using-Gensim-Embeddings-with-Keras-and-Tensorflow)) is to build the `Embedding` layer manually from the trained vectors:


In [ ]:
def gensim_to_keras_embedding(w2v_model, train_embeddings=False):
    """Build a tf.keras Embedding layer initialized with a trained gensim Word2Vec/FastText
    model's vectors, correctly aligning Keras's word-index scheme with gensim's vocabulary."""
    from tensorflow.keras.layers import Embedding

    keyed_vectors = w2v_model.wv
    weights = keyed_vectors.vectors
    index_to_key = keyed_vectors.index_to_key

    layer = Embedding(
        input_dim=weights.shape[0],
        output_dim=weights.shape[1],
        weights=[weights],
        trainable=train_embeddings,
        name='w2v_embedding',
    )
    return layer, {word: i for i, word in enumerate(index_to_key)}

embedding_layer, word_to_index = gensim_to_keras_embedding(w2v_model, train_embeddings=False)
print(embedding_layer)
print("Example index lookup - 'story' ->", word_to_index.get('story'))


### **5. FastText Vectorization** 

Word2Vec has one significant blind spot: if a word never appeared in training, it has **no vector at all** — `model.wv['fantastical']` raises a `KeyError` if `fantastical` wasn't in the training corpus, even though `fantastic` was.

**FastText** (Bojanowski et al., 2017, from Facebook AI Research) fixes this by learning vectors for **character n-grams** (subword pieces) instead of only whole words, then representing each word as the sum of its subword vectors. A word never seen during training can still get a reasonable vector, built from the subword pieces it shares with words that *were* seen — including handling typos and rare morphological variants (`"boringly"`, `"unboring"`, etc.) much more gracefully than Word2Vec.


In [ ]:
from gensim.models import FastText

ft_model = FastText(
    tokenized_docs,
    vector_size=8,
    window=3,
    min_count=1,
    sg=0,
    workers=1,
)

# The key demo: ask both models for a word that is NOT in the training vocabulary.
oov_word = 'fantastical'  # note: 'fantastic' is in our corpus, 'fantastical' deliberately is not

print(f"Is '{oov_word}' in the training vocabulary? "
      f"{oov_word in [t for doc in tokenized_docs for t in doc]}")

try:
    w2v_model.wv[oov_word]
    print("Word2Vec: produced a vector (unexpected for an unseen word)")
except KeyError:
    print(f"Word2Vec: KeyError - '{oov_word}' has no vector, it was never seen during training.")

try:
    vec = ft_model.wv[oov_word]
    print(f"FastText: produced an {vec.shape[0]}-dim vector for '{oov_word}' anyway, "
          f"built from its subword pieces (e.g. 'fantas', 'antastic', 'tic', ...).")
except KeyError:
    print(f"FastText: also could not produce a vector for '{oov_word}'.")


This is the single biggest practical reason to reach for FastText over plain Word2Vec: real-world text is full of typos, rare inflections, and words your training corpus simply never happened to contain, and FastText degrades gracefully in exactly those cases instead of failing outright.


### **6. Pretrained Embeddings: GloVe** 

Word2Vec and FastText above were trained **from scratch on our tiny 8-sentence corpus** — nowhere near enough text to learn real semantics. In practice, it's far more common to use **pretrained embeddings**: vectors trained once on a massive corpus (billions of words) and then reused for any downstream task, the same way you'd reuse a pretrained image model instead of training a CNN from scratch on a handful of photos.

**GloVe** (Pennington et al., 2014, Stanford) is one of the most widely used pretrained embedding sets, trained on corpora like Wikipedia + Gigaword. `gensim`'s downloader API can fetch a pretrained GloVe model directly (**this cell needs internet access** the first time it runs, to download the vectors — after that they're cached locally).


In [ ]:
import gensim.downloader as api

# 'glove-wiki-gigaword-50' = 50-dimensional GloVe vectors trained on Wikipedia + Gigaword.
# Larger, more accurate options also exist (glove-wiki-gigaword-100/200/300), at the cost of a
# bigger download.
glove = api.load('glove-wiki-gigaword-50')


In [ ]:
# Real semantic similarity, learned from billions of words - contrast this with our
# toy Word2Vec model's much noisier results earlier.
glove.most_similar('movie')


**The famous demo of what these vectors actually capture — vector arithmetic on meaning itself:**

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

If embeddings truly capture semantic relationships, subtracting the "male" direction from `king` and adding the "female" direction should land near `queen` — without ever being told what a queen is.


In [ ]:
result = glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)
pd.DataFrame(result, columns=['word', 'cosine similarity'])


This is the payoff of the entire progression in this notebook: Bag-of-Words can only tell you *whether* a word appeared; TF-IDF can tell you *how distinctive* a word is; but only dense, learned embeddings like Word2Vec/FastText/GloVe capture that `king`, `queen`, `man`, and `woman` sit in a meaningful geometric relationship to each other.


## A note on where this goes next: contextual embeddings

Every technique above — including Word2Vec, FastText, and GloVe — gives each word **exactly one fixed vector**, regardless of context. `"bank"` gets the same vector whether it means a river bank or a financial bank, which is a real limitation.

The current state of the art, **contextual embeddings** from Transformer models like **BERT**, produce a *different* vector for the same word depending on the sentence it appears in, by processing the whole sequence at once with self-attention. That's a substantially bigger topic (and typically needs a GPU and a `transformers`-style library) — worth flagging as the natural "next lesson" after this one, but out of scope for a notebook focused on classical vectorization techniques.


## Summary: choosing a vectorization technique

| Technique | Sparse/Dense | Captures similarity? | Handles unseen words? | Typical use case |
|---|---|---|---|---|
| Count Vectorization | Sparse | No | No (unseen word = ignored) | Simple baselines, small vocabularies |
| N-grams | Sparse | Partially (local phrases) | No | Capturing negation/short phrases cheaply |
| TF-IDF | Sparse, weighted | No | No | Search/keyword extraction, classic text classification baselines |
| Word2Vec | Dense | Yes (if enough training data) | No | Custom domain vocabulary, when you can train on a large in-domain corpus |
| FastText | Dense (subword) | Yes | Yes (via subwords) | Morphologically rich languages, noisy/typo-prone text |
| GloVe (pretrained) | Dense, pretrained | Yes, strongly | Only for words in its vocabulary | Quick strong baseline without training your own embeddings |
| *(BERT / contextual)* | Dense, contextual | Yes, context-sensitive | Yes | Current state of the art; more compute-hungry |

**Rule of thumb for a first project:** start with TF-IDF — it's fast, interpretable, and a surprisingly strong baseline for many text classification tasks. Reach for Word2Vec/FastText/GloVe when you specifically need to capture word *meaning* and *similarity*, and reach for contextual embeddings (BERT and friends) when the same word's meaning genuinely changes by context in your task.
